# Day 3c. Skills: install the training

Every chat is the agent's first day at work. A **skill** is the onboarding
binder: a folder with a `SKILL.md` contract that says *when* to act and *how you
do it*.

A skill is a **document, not code**, which means the person who knows the
procedure can write one, and it ships to every agent that mounts the folder.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below, e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

### Restore yesterday's world

Every day-3 notebook starts from the same booking agent you built on day 2.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool

HOTELS = {"Hotel Anders":  {"free": True,  "eur": 92},
          "Pension Krumm": {"free": True,  "eur": 61},
          "Grand Mitte":   {"free": False, "eur": 210}}

@tool
def check_availability(hotel: str) -> dict:
    """Live availability and price (EUR/night) for one hotel."""
    return HOTELS.get(hotel, {"error": f"unknown hotel: {hotel}"})

@tool
def list_hotels() -> list:
    """All hotels we can book, cheapest first."""
    return sorted(HOTELS, key=lambda h: HOTELS[h]["eur"])

@tool
def book_room(hotel: str, nights: int) -> dict:
    """Book a room. THIS ONE WRITES."""
    return {"confirmation": f"GIU-{abs(hash(hotel + str(nights))) % 10000:04d}"}

INSTRUCTIONS = ("You are the GIU booking assistant. "
                "Check availability before booking. Prices in EUR.")
print("world restored")

## 1 · Write two skills

The contract: YAML frontmatter with `name` and `description` (the **trigger**,
that is how the agent decides this skill matches), then the instructions.

In [ ]:
import pathlib, textwrap

def make_skill(name, description, body):
    p = pathlib.Path("skills") / name
    p.mkdir(parents=True, exist_ok=True)
    (p / "SKILL.md").write_text(textwrap.dedent(f"""\
    ---
    name: {name}
    description: >-
      {description}
    ---
    {body}"""))

make_skill("booking-report",
    "Use when a booking should be summarized into the official trip-report format.",
    """# Instructions
    1. Gather: hotel, nights, price per night, confirmation code.
    2. Output EXACTLY this format, nothing else:
       TRIP REPORT. Berlin ([nights] nights)
       Hotel: [hotel]. EUR [price]/night, ref [code]
       Total: EUR [nights x price]
    3. No greetings, no closing line.""")

make_skill("polite-decline",
    "Use when a request is out of scope for the booking assistant: flights, visas, refunds.",
    """# Instructions
    1. Decline in ONE sentence, warmly.
    2. Point to the official travel portal: travel.giu.example
    3. Never apologize more than once.""")

print(*pathlib.Path("skills").rglob("SKILL.md"), sep="\n")
print()
print(pathlib.Path("skills/booking-report/SKILL.md").read_text())

## 2 · The injector, a middleware you can read

Three moves, and you know all of them from 3a:

1. **scan** `skills/` and parse each frontmatter
2. at `wrap_model_call`, append the **menu** (names + descriptions only) to the
   system prompt
3. expose a `read_skill` tool so the agent pulls the **body** when one matches

That is progressive disclosure: cheap metadata always, expensive instructions
only on demand.

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from langchain.messages import SystemMessage

def scan_skills(root="skills"):
    out = {}
    for f in pathlib.Path(root).glob("*/SKILL.md"):
        meta, lines = {}, f.read_text().splitlines()
        for line in lines[1:lines.index("---", 1)]:      # the frontmatter block
            if ":" in line:
                k, _, v = line.partition(":")
                meta[k.strip()] = v.strip().lstrip(">-").strip()
        out[meta.get("name", f.parent.name)] = {
            "description": meta.get("description", ""), "path": f}
    return out

@tool
def read_skill(name: str) -> str:
    """Read the full instructions of one available skill by name."""
    s = scan_skills().get(name)
    return s["path"].read_text() if s else f"unknown skill: {name}"

class SkillsMiddleware(AgentMiddleware):
    tools = (read_skill,)                # the middleware brings its own tool

    def wrap_model_call(self, request, handler):
        menu = "\n".join(f"- {n}: {s['description']}"
                          for n, s in scan_skills().items())
        blocks = list(request.system_message.content_blocks)
        blocks.append({"type": "text", "text":
            "\n## Skills available (call read_skill when one matches):\n" + menu})
        return handler(request.override(
            system_message=SystemMessage(content=blocks)))

print("SkillsMiddleware ready -", len(scan_skills()), "skills discovered")

In [ ]:
skilled = create_agent(model=llm,
    tools=[check_availability, list_hotels, book_room],
    system_prompt=INSTRUCTIONS,
    middleware=[SkillsMiddleware()])

out = skilled.invoke({"messages": [HumanMessage(
    "Book the cheapest room for 2 nights, then give me the trip report.")]})

for m in out["messages"]:                # watch it pick the skill
    body = m.content if m.content else getattr(m, "tool_calls", "")
    print(m.type.upper().ljust(6), "→", str(body)[:110])

In that trace: the agent called `read_skill("booking-report")`, because the
*description* matched the task, and then followed the format exactly.

Now ask it something out of scope and watch the other skill fire:

In [ ]:
out = skilled.invoke({"messages": [HumanMessage(
    "Can you also book my flight and sort out my visa?")]})
print(out["messages"][-1].content)

## 3 · The reveal: this already ships

What you just wrote is `SkillsMiddleware`, and a hardened version of it already
exists in the **deepagents** package. Same idea, same progressive disclosure,
years of edge cases handled.

And it is still **the same `create_agent` call you have used all week**. You are
not switching framework, you are importing a better middleware:

```python
from deepagents.middleware import SkillsMiddleware, FilesystemMiddleware
```

Two small notes on the arguments:

- `sources=["./skills/"]` is the folder to scan, same as our `scan_skills()`.
- `backend=` is *where the skill files are read from*. We point it at the real
  folder on disk. Backends are the whole of session 3d, so for now just accept
  that this is how the middleware reaches the files.
- The shipped middleware reads skills with a `read_file` tool rather than our
  `read_skill`, and that tool comes from `FilesystemMiddleware`. So we add both.

In [ ]:
from deepagents.middleware import SkillsMiddleware, FilesystemMiddleware
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(root_dir=".", virtual_mode=False)   # more in 3d

shipped = create_agent(              # ← still create_agent
    model=llm,
    tools=[check_availability, book_room],
    system_prompt=INSTRUCTIONS,
    middleware=[
        FilesystemMiddleware(backend=backend),        # gives it read_file
        SkillsMiddleware(backend=backend,             # gives it the menu
                         sources=["./skills/"]),
    ])

out = shipped.invoke({"messages": [HumanMessage(
    "I stayed at Pension Krumm, 2 nights, 61 EUR/night, ref GIU-4417. "
    "Give me the trip report.")]})

for m in out["messages"]:
    body = m.content if m.content else getattr(m, "tool_calls", "")
    print(m.type.upper().ljust(6), "→", str(body)[:110])

Read that trace next to your own one. Same shape: a menu in the system prompt, a
file read when the task matches, then the format followed.

One detail worth seeing. The shipped middleware writes its own instructions into
the system prompt, and they say this in plain words:

> **How to Use Skills (Progressive Disclosure):** Skills follow a progressive
> disclosure pattern. You see their name and description above, but only read
> full instructions when needed.

The pattern has a name, and the library uses it too.

Build it once to understand it, then take the shipped one. That order is the
habit that transfers to every framework you will meet.

## 4 · Your turn, write a skill

No Python. Author a **third skill** and watch behavior change without touching
a single line of agent code.

Ideas: `expense-summary` (turn a booking into an expense line), `email-draft`
(a confirmation email in your university's tone), `compare-hotels` (a fixed
comparison table format).

Then re-run the agent, our middleware re-scans on every call, so it picks the
new skill up with no restart.

In [ ]:
make_skill("expense-summary",
    "YOUR description here, this is the trigger, write it carefully",
    """# Instructions
    1. ...
    2. ...""")

out = skilled.invoke({"messages": [HumanMessage("... your test request ...")]})
print(out["messages"][-1].content)

**Then discuss:**
- Your description decides whether the skill is ever used. Rewrite it badly on
  purpose, does the agent stop finding it?
- `scan_skills()` runs on **every model call**. What does that cost, and what
  would you cache?

---
## Wrap

A skill is a folder with a contract · the menu rides in the system prompt, the
body loads on demand · one `wrap_model_call` hook injects it · deepagents ships
the hardened version.

**Next (3d):** the agent writes files and runs code. Where does that land?